In [1]:
import os
print(os.getcwd())

/content


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 2. 현재 작업 디렉토리를 'model' 폴더로 변경
# 'KOPIS 공모전' 폴더가 내 드라이브 바로 아래에 있다고 가정합니다.
os.chdir('/content/drive/MyDrive/KOPIS 공모전/model')

# 현재 작업 디렉토리 확인 (이제 'model' 폴더가 되어야 합니다)
print(os.getcwd())

/content/drive/MyDrive/KOPIS 공모전/model


## 최종

In [6]:
!pip uninstall torch -y
!pip uninstall torch-geometric -y
!pip install torch torch-geometric

Found existing installation: torch 2.8.0+cu126
Uninstalling torch-2.8.0+cu126:
  Successfully uninstalled torch-2.8.0+cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 8.4 MB/s eta 0:00:00
 

In [24]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import torch.optim as optim
import re
from torch_geometric.utils import to_undirected, add_self_loops
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import os

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 사용 가능한 장치: {device}")

# =========================
# 1. 데이터 로드
# =========================
try:
    df = pd.read_csv('../data/except_complex.csv', encoding='utf-8-sig')
    df_complex = pd.read_csv('../data/complex.csv', encoding='utf-8-sig')
except FileNotFoundError:
    print("파일을 찾을 수 없습니다. 예시 데이터로 대체합니다.")
    df_data = {
        '공연명': ['보디가드', '인어공주', '허클베리 핀', '절대 변하지 않는 것들', '서커스 쇼'],
        '장르명': ['뮤지컬', '뮤지컬', '대중음악', '국악(국악)', '서커스/마술'],
        '출연진내용': ['김선영, 박기영', '김우현, 김호경', 'NaN', '안은경, 전우석', '삐에로'],
        '제작진내용': ['박소영, 박지훈', '손지현, 김현주', 'NaN', '류형선, 정우연', 'NaN'],
        '세부장르명': ['NaN','NaN','NaN','NaN','NaN']
    }
    df_complex_data = {
        '공연명': ['인더스트리얼 퍼포먼스','아웃 오버 블루','서커스 콘서트'],
        '장르명': ['복합','복합','서커스/마술'],
        '출연진내용': ['예효송, 나혜영','신유, 김지연','곡예사'],
        '제작진내용': ['NaN','류경한','NaN'],
        '세부장르명': ['다원/융복합','복합','NaN']
    }
    df = pd.DataFrame(df_data)
    df_complex = pd.DataFrame(df_complex_data)

현재 사용 가능한 장치: cuda


In [25]:
# =========================
# 2. 필터링
# =========================
df = df[df['장르명'] != '서커스/마술'].reset_index(drop=True)
df_complex = df_complex[df_complex['장르명'] != '서커스/마술'].reset_index(drop=True)
df_complex = df_complex[df_complex['세부장르명'] != '다원/융복합'].reset_index(drop=True)

# 🔥 중복 컬럼 제거 + original_index 재생성
df = df.loc[:, ~df.columns.duplicated()].reset_index(drop=True).reset_index().rename(columns={'index':'original_index'})
df['source'] = 'df'

df_complex = df_complex.loc[:, ~df_complex.columns.duplicated()].reset_index(drop=True).reset_index().rename(columns={'index':'original_index'})
df_complex['source'] = 'df_complex'

print("CHECK df:", df[['공연명','original_index','source']].head())
print("CHECK df_complex:", df_complex[['공연명','original_index','source']].head())


CHECK df:             공연명  original_index source
0          보디가드               0     df
1     인어공주 [구미]               1     df
2   허클베리피: 분신 9               2     df
3  절대 변하지 않는 것들               3     df
4       난타 [명동]               4     df
CHECK df_complex:                               공연명  original_index      source
0                      아웃 오브 더 블루               0  df_complex
1                           이색콘서트               1  df_complex
2                차세대열전, 이어지음 프로젝트               2  df_complex
3         헨젤과 그레텔, 토선생 용궁가다 (패키지)               3  df_complex
4  최정주의 아트앤뮤직 큐레이션: 20세기를 빛낸 화가들Ⅱ               4  df_complex


In [26]:
# =========================
# 3. 이름 표준화
# =========================
def split_and_clean(text):
    if pd.isna(text): return []
    names = text.replace(' ', '').split(',')
    names = [re.sub(r'\(.*?\)| 등| 외| 外', '', n).strip() for n in names]
    return [n for n in names if n]

df['출연진내용_standardized'] = df['출연진내용'].apply(split_and_clean)
df['제작진내용_standardized'] = df['제작진내용'].apply(split_and_clean)
df_complex['출연진내용_standardized'] = df_complex['출연진내용'].apply(split_and_clean)
df_complex['제작진내용_standardized'] = df_complex['제작진내용'].apply(split_and_clean)

print("데이터 전처리 완료!")

데이터 전처리 완료!


In [27]:

# =========================
# 4. 노드 딕셔너리 생성
# =========================
node_dict = {}
current_id = 0
all_performances = pd.concat([df, df_complex], ignore_index=True)

# 인물 노드
for _, row in all_performances.iterrows():
    for name in row['출연진내용_standardized']:
        k = f"출연진_{name}"
        if k not in node_dict:
            node_dict[k] = current_id; current_id += 1
    for name in row['제작진내용_standardized']:
        k = f"제작진_{name}"
        if k not in node_dict:
            node_dict[k] = current_id; current_id += 1

# 공연 노드
for _, row in all_performances.iterrows():
    k = f"공연_{row['source']}_{int(row['original_index'])}"
    if k not in node_dict:
        node_dict[k] = current_id; current_id += 1

# 장르 노드
genre_names = sorted(list(df['장르명'].apply(lambda x: re.sub(r'\s*\(.*?\)','',str(x)).strip()).unique()))
for g in genre_names:
    k = f"장르_{g}"
    if k not in node_dict:
        node_dict[k] = current_id; current_id += 1

num_nodes = len(node_dict)

In [28]:
# =========================
# 5. 엣지 생성 (공연-인물만)
# =========================
edges = []
for _, row in all_performances.iterrows():
    perf_key = f"공연_{row['source']}_{int(row['original_index'])}"
    perf_id = node_dict[perf_key]
    for n in row['출연진내용_standardized']:
        if f"출연진_{n}" in node_dict:
            edges.append([perf_id, node_dict[f"출연진_{n}"]])
    for n in row['제작진내용_standardized']:
        if f"제작진_{n}" in node_dict:
            edges.append([perf_id, node_dict[f"제작진_{n}"]])

edges_array = np.array(edges, dtype=np.int64)
edge_index_full = torch.tensor(edges_array.T, dtype=torch.long)
edge_index_full = to_undirected(edge_index_full, num_nodes=num_nodes)
edge_index_full, _ = add_self_loops(edge_index_full, num_nodes=num_nodes)
edge_index_full = edge_index_full.to(device)


In [29]:

# =========================
# 6. 라벨 및 마스크
# =========================
genre_id_labels = {g:i for i,g in enumerate(genre_names)}
y = torch.full((num_nodes,), -1, dtype=torch.long).to(device)
train_mask = torch.zeros(num_nodes, dtype=torch.bool).to(device)
predict_mask = torch.zeros(num_nodes, dtype=torch.bool).to(device)

# train mask
for _, row in df.iterrows():
    perf_key = f"공연_{row['source']}_{int(row['original_index'])}"
    if perf_key in node_dict:
        pid = node_dict[perf_key]
        gname = re.sub(r'\s*\(.*?\)','',str(row['장르명'])).strip()
        if gname in genre_id_labels:
            y[pid] = genre_id_labels[gname]
            train_mask[pid] = True

# predict mask
hits, misses, missing_keys = 0,0,[]
for _, row in df_complex.iterrows():
    perf_key = f"공연_{row['source']}_{int(row['original_index'])}"
    if perf_key in node_dict:
        predict_mask[node_dict[perf_key]] = True
        hits += 1
    else:
        misses += 1
        missing_keys.append(perf_key)

print(f"[CHECK] df_complex 매칭 성공: {hits}, 실패: {misses}")
if misses:
    print("예시 missing keys:", missing_keys[:5])

# 최종 Data 객체
data = Data(x=None, edge_index=edge_index_full, y=y).to(device)
data.train_mask = train_mask
data.predict_mask = predict_mask
data.num_nodes = num_nodes

print("GNN Data 객체 구축 완료!")
print(f"총 노드 수: {data.num_nodes}, 총 엣지 수: {data.edge_index.size(1)}")
print(f"학습용 공연 노드 수: {data.train_mask.sum()}, 예측용 공연 노드 수: {data.predict_mask.sum()}")

[CHECK] df_complex 매칭 성공: 1099, 실패: 0
GNN Data 객체 구축 완료!
총 노드 수: 138935, 총 엣지 수: 696475
학습용 공연 노드 수: 72612, 예측용 공연 노드 수: 1099


In [30]:

# --- Train / Validation 분리 ---
all_perf_train_nodes = np.where(data.train_mask.cpu().numpy())[0]  # except_complex 공연 노드
train_idx, val_idx = train_test_split(all_perf_train_nodes, test_size=0.2, random_state=42)

train_mask_split = torch.zeros_like(data.train_mask)
val_mask_split   = torch.zeros_like(data.train_mask)
train_mask_split[torch.tensor(train_idx, device=train_mask_split.device)] = True
val_mask_split[torch.tensor(val_idx,   device=val_mask_split.device)]   = True

# (핵심) 학습 그래프에서 '검증 공연 노드'가 포함된 엣지를 제거
# 이렇게 해야 검증 데이터의 라벨 정보가 학습에 유출되는 것을 방지할 수 있습니다.
ei = edge_index_full.t().cpu().numpy()               # [E, 2]
val_set = set(val_idx.tolist())
# 엣지의 양 끝 노드 중 하나라도 검증 노드에 속하면 제거
keep = np.logical_and(~np.isin(ei[:,0], list(val_set)),
                      ~np.isin(ei[:,1], list(val_set)))
edge_index_train = torch.tensor(ei[keep].T, dtype=torch.long, device=device)

print(f"학습용 엣지 수 (검증 엣지 제거 후): {edge_index_train.size(1)}")

학습용 엣지 수 (검증 엣지 제거 후): 572724


In [31]:

# --- GNN 모델 정의 및 학습 (GPU) ---

# 4-1. GNN 모델 정의 (RAM & GPU 최적화)
class GCN(nn.Module):
    def __init__(self, num_nodes, num_genres, embedding_dim=64, hidden_channels=64):
        super().__init__()
        # 노드 피처를 Embedding 레이어에서 생성
        self.node_embedding = nn.Embedding(num_embeddings=num_nodes, embedding_dim=embedding_dim)
        self.conv1 = GCNConv(embedding_dim, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, num_genres)

    def forward(self, edge_index):
        # 노드 임베딩 생성 (모든 노드에 대해)
        x = self.node_embedding(torch.arange(self.node_embedding.num_embeddings, device=edge_index.device))
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# 4-2. 모델 인스턴스화 및 학습
num_genres = len(genre_names)
model = GCN(num_nodes, num_genres, embedding_dim=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.NLLLoss()

print("\n모델 학습 시작...")
model.train()
epochs = 200 # 에폭 수 감소 (예시)
for epoch in range(epochs):
    optimizer.zero_grad()
    # 학습은 검증 엣지가 제거된 그래프로 진행
    out = model(edge_index_train)

    # 손실 계산은 학습 마스크가 적용된 노드에 대해서만
    loss = criterion(out[train_mask_split], data.y[train_mask_split])

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0: # 20 에폭마다 출력
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {loss.item():.4f}')

print("\n모델 학습 완료!")



모델 학습 시작...
Epoch 20/200, Train Loss: 1.0491
Epoch 40/200, Train Loss: 0.2520
Epoch 60/200, Train Loss: 0.0496
Epoch 80/200, Train Loss: 0.0175
Epoch 100/200, Train Loss: 0.0085
Epoch 120/200, Train Loss: 0.0059
Epoch 140/200, Train Loss: 0.0041
Epoch 160/200, Train Loss: 0.0032
Epoch 180/200, Train Loss: 0.0028
Epoch 200/200, Train Loss: 0.0024

모델 학습 완료!


In [32]:

# --- 평가 및 예측 ---

model.eval()
with torch.no_grad():
    # 평가 및 예측은 전체 그래프 (출연/제작 엣지만 포함)로 진행
    out = model(data.edge_index)
    probs = torch.exp(out)  # log_softmax → 확률

    # ===========================
    # 1) except_complex (Validation) 평가
    # ===========================
    # 검증 마스크가 적용된 노드에 대해 평가
    val_true = data.y[val_mask_split].cpu().numpy()
    val_pred = probs[val_mask_split].argmax(dim=1).cpu().numpy()

    acc = accuracy_score(val_true, val_pred)
    f1 = f1_score(val_true, val_pred, average="macro")

    # Top-2 Accuracy (검증 데이터에 대해 계산)
    top2_preds = torch.topk(probs[val_mask_split], k=2, dim=1).indices.cpu().numpy()
    hits = sum(val_true[i] in top2_preds[i] for i in range(len(val_true)))
    top2_acc = hits / len(val_true)

    print("\n[Validation 평가 결과]")
    print(f"Accuracy: {acc:.4f}, Macro-F1: {f1:.4f}, Top-2 Accuracy: {top2_acc:.4f}")

    # ===========================
    # 2) complex 공연 예측
    # ===========================
    # 예측 마스크가 적용된 노드에 대해 예측
    complex_probs = probs[data.predict_mask]
    top2_probs, top2_indices = torch.topk(complex_probs, k=2, dim=1)

# 결과 역매핑 및 저장
reverse_genre_map = {v: k for k, v in genre_id_labels.items()}
reverse_node_map  = {v: k for k, v in node_dict.items()}
complex_perf_nodes = np.where(data.predict_mask.cpu().numpy())[0] # complex 공연 노드의 인덱스

results = []

print("\n[복합 장르 공연 예측 결과 (Top-2 조합)]")
for i, node_id in enumerate(complex_perf_nodes):
    node_key = reverse_node_map[node_id]              # 예: "공연_df_complex_1220"
    parts = node_key.split('_')
    source = "_".join(parts[1:-1])                    # ex) "df_complex"
    original_index = int(parts[-1])                   # 마지막 숫자

    # 원래 데이터프레임에서 공연명 찾기
    performance_info = all_performances[
        (all_performances['source'] == source) &
        (all_performances['original_index'] == original_index)
    ]

    if not performance_info.empty:
        performance_name = performance_info['공연명'].iloc[0]

        # 상위 2개 장르 및 확률
        genres = [reverse_genre_map[idx] for idx in top2_indices[i].cpu().numpy()]
        probs_vals = top2_probs[i].cpu().numpy()
        combined = "+".join(genres)

        # 결과 dict 형태로 저장
        results.append({
            "공연명": performance_name,
            "예측 복합 장르": combined,
            "1위 장르": genres[0],
            "1위 확률": round(float(probs_vals[0]), 4),
            "2위 장르": genres[1],
            "2위 확률": round(float(probs_vals[1]), 4),
        })
    else:
         print(f"Warning: Could not find performance info for node key '{node_key}'")


# DataFrame 생성
df_results = pd.DataFrame(results)

# 결과 출력 (첫 10개만 예시로 출력)
print(df_results.head(10).to_markdown(index=False))

# CSV 저장 (utf-8-sig로 하면 한글 깨짐 방지)
output_path = os.path.join(os.getcwd(), "complex_genre_predictions.csv")
df_results.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n예측 결과를 CSV 파일로 저장했습니다: {output_path}")



[Validation 평가 결과]
Accuracy: 0.6621, Macro-F1: 0.5013, Top-2 Accuracy: 0.7690

[복합 장르 공연 예측 결과 (Top-2 조합)]
| 공연명                                              | 예측 복합 장르    | 1위 장르   |   1위 확률 | 2위 장르   |   2위 확률 |
|:----------------------------------------------------|:------------------|:-----------|-----------:|:-----------|-----------:|
| 아웃 오브 더 블루                                   | 서양음악+한국음악 | 서양음악   |     1      | 한국음악   |     0      |
| 이색콘서트                                          | 서양음악+뮤지컬   | 서양음악   |     0.7545 | 뮤지컬     |     0.2453 |
| 차세대열전, 이어지음 프로젝트                       | 무용+한국음악     | 무용       |     0.8246 | 한국음악   |     0.1448 |
| 헨젤과 그레텔, 토선생 용궁가다 (패키지)             | 한국음악+서양음악 | 한국음악   |     0.9981 | 서양음악   |     0.0012 |
| 최정주의 아트앤뮤직 큐레이션: 20세기를 빛낸 화가들Ⅱ | 서양음악+무용     | 서양음악   |     1      | 무용       |     0      |
| 최정주의 아트앤뮤직 큐레이션: 20세기를 빛낸 화가들Ⅲ | 서양음악+무용     | 서양음악   |     1      | 무용       |     0      |
| LED비바츠 태권발레 [철원]                           | 대중무용+뮤지

# 추가
1. 멀티 라벨링
2. 클래스 가중치 적용
3. 출연/제작 인물 TF-IDF 가중치 적용

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

# =========================
# 1. 멀티라벨 라벨 생성
# =========================
num_genres = len(genre_names)
y = torch.zeros((num_nodes, num_genres), dtype=torch.float).to(device)
train_mask = torch.zeros(num_nodes, dtype=torch.bool).to(device)
predict_mask = torch.zeros(num_nodes, dtype=torch.bool).to(device)

for _, row in df.iterrows():
    perf_key = f"공연_{row['source']}_{int(row['original_index'])}"
    if perf_key in node_dict:
        pid = node_dict[perf_key]
        genre_name = re.sub(r'\s*\(.*?\)','',str(row['장르명'])).strip()
        if genre_name in genre_id_labels:
            y[pid, genre_id_labels[genre_name]] = 1.0
            train_mask[pid] = True

for _, row in df_complex.iterrows():
    perf_key = f"공연_{row['source']}_{int(row['original_index'])}"
    if perf_key in node_dict:
        predict_mask[node_dict[perf_key]] = True

# =========================
# 2. 클래스 가중치 (inverse frequency)
# =========================
genre_counts = y[train_mask].sum(dim=0).cpu().numpy()
class_weights = 1.0 / (genre_counts + 1e-6)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# =========================
# 3. TF-IDF 기반 edge_weight
# =========================
# 인물 노드 이름 추출
person_nodes = [n for n in node_dict if n.startswith("출연진_") or n.startswith("제작진_")]

# 각 공연마다 인물 문자열 합치기 → TF-IDF 벡터화
corpus = []
for _, row in all_performances.iterrows():
    text = " ".join(row['출연진내용_standardized'] + row['제작진내용_standardized'])
    corpus.append(text)

vectorizer = TfidfVectorizer(token_pattern=r"[^ ]+")
tfidf = vectorizer.fit_transform(corpus)

# edge_weight 매핑
edge_weights = []
for e in edges:
    perf_id, person_id = e
    # 공연 키 ↔ index 매핑
    perf_key = [k for k,v in node_dict.items() if v == perf_id][0]
    perf_idx = all_performances.index[all_performances.apply(
        lambda r: f"공연_{r['source']}_{int(r['original_index'])}" == perf_key, axis=1
    )].tolist()[0]

    person_key = [k for k,v in node_dict.items() if v == person_id][0]
    person_name = person_key.split("_",1)[1]

    col_idx = vectorizer.vocabulary_.get(person_name, None)
    if col_idx is not None:
        weight = float(tfidf[perf_idx, col_idx])  # sparse → float
    else:
        weight = 0.0

    edge_weights.append(weight)

edge_weight = torch.tensor(edge_weights, dtype=torch.float).to(device)

# =========================
# 4. 모델 정의 (멀티라벨 + edge_weight)
# =========================
class GCN_Multi(nn.Module):
    def __init__(self, num_nodes, num_genres, embedding_dim=64, hidden_channels=64):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        self.conv1 = GCNConv(embedding_dim, hidden_channels, normalize=False)
        self.conv2 = GCNConv(hidden_channels, num_genres, normalize=False)

    def forward(self, edge_index, edge_weight):
        x = self.node_emb(torch.arange(self.node_emb.num_embeddings, device=edge_index.device))
        x = self.conv1(x, edge_index, edge_weight).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index, edge_weight)
        return x  # logits (no softmax)

model = GCN_Multi(num_nodes, num_genres).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)

# =========================
# 5. 학습 루프
# =========================
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    out = model(edge_index, edge_weight)
    loss = criterion(out[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# =========================
# 6. 평가 (멀티라벨)
# =========================
model.eval()
with torch.no_grad():
    logits = model(edge_index, edge_weight)
    probs = torch.sigmoid(logits)

    preds = (probs > 0.5).float()

    acc = accuracy_score(y[train_mask].cpu().numpy(), preds[train_mask].cpu().numpy())
    f1 = f1_score(y[train_mask].cpu().numpy(), preds[train_mask].cpu().numpy(), average='macro')

print(f"[Train Evaluation] Accuracy: {acc:.4f}, Macro-F1: {f1:.4f}")


## 최최종